In [1]:
import os, sys
from pathlib import Path
import numpy as np
import torch
# 路径设置
HERE = Path(os.getcwd()).resolve()
if HERE.name == "fit-4hdnnp-NaCl-initial":
    ROOT_DIR = HERE.parent
    DATA_DIR = HERE
else:
    ROOT_DIR = HERE
    DATA_DIR = ROOT_DIR / "fit-4hdnnp-NaCl-initial"
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
print("ROOT_DIR:", ROOT_DIR)
print("DATA_DIR:", DATA_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

ROOT_DIR: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq
DATA_DIR: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq/fit-4hdnnp-NaCl-initial
device: cpu


In [2]:
import cace
from cace.representations import Cace
from cace.modules import BesselRBF, PolynomialCutoff
from cace.tools.scatter import scatter_sum
from cace.tools import torch_geometric
from cace.models.atomistic import NeuralNetworkPotential
from cace.data.extxyz_charge import get_dataset_from_extxyz_with_charge
from cace.modules import ChargeEq
from cace.tools import Metrics
cutoff = 5.29
Fourier_node = 18
# 与训练脚本一致的表示
radial_basis = BesselRBF(cutoff=cutoff, n_rbf=6, trainable=True)
cutoff_fn = PolynomialCutoff(cutoff=cutoff)
rep = Cace(
    zs=[11, 17],
    n_atom_basis=2,
    embed_receiver_nodes=True,
    cutoff=cutoff,
    cutoff_fn=cutoff_fn,
    radial_basis=radial_basis,
    n_radial_basis=8,
    max_l=3,
    max_nu=3,
    num_message_passing=0,
    type_message_passing=["Bchi"],
    args_message_passing={"Bchi": {"shared_channels": False, "shared_l": False}},
    device=device,
    timeit=False,
    forward_features=["atomic_numbers"],
).to(device)
# 短程能量
sr_energy = cace.modules.atomwise.Atomwise(
    n_layers=3,
    output_key="SR_energy",
    n_hidden=[32, 16],
    use_batchnorm=False,
    add_linear_nn=True,
)
# chi
chi = cace.modules.Atomwise(
    n_layers=3,
    n_hidden=[24, 12],
    n_out=1,
    per_atom_output_key="chi",
    output_key="tot_chi",
    residual=False,
    add_linear_nn=True,
    post_process=torch.square,
    bias=False,
)
# system_charge
class SystemChargeFromAtomicCharges(torch.nn.Module):
    def __init__(self, charges_key: str = "charge", output_key: str = "system_charge"):
        super().__init__()
        self.charges_key = charges_key
        self.output_key = output_key
        self.model_outputs = [output_key]
    def forward(self, data: dict, **kwargs):
        if self.charges_key not in data or data[self.charges_key] is None:
            if data.get("batch", None) is None:
                num_graphs = 1
            else:
                num_graphs = int(data["batch"].max().item()) + 1 if data["batch"].numel() > 0 else 1
            data[self.output_key] = torch.zeros(
                (num_graphs,), device=data["positions"].device, dtype=data["positions"].dtype
            )
            return data
        q = data[self.charges_key]
        if q.dim() > 1:
            q = q.view(-1)
        if data.get("batch", None) is None:
            system_q = q.sum().view(1)
        else:
            system_q = scatter_sum(q, data["batch"], dim=0)
        data[self.output_key] = system_q
        return data
system_charge_from_q = SystemChargeFromAtomicCharges("charge", "system_charge")
# ChargeEq
charge_eq = cace.modules.ChargeEq(
    dl=1.5,
    sigma=1.0,
    elements=[11, 17],
    feature_key="chi",
    output_key="q_eq",
    ewald_key="SOG_potential",
    system_charge=None,
    remove_self_interaction=True,
    aggregation_mode="sum",
    use_sog_kernel=True,
    sog_num_components=Fourier_node,
)
# 总能量
e_add = cace.modules.FeatureAdd(
    feature_keys=["SR_energy", "SOG_potential"],
    output_key="CACE_energy",
)
# Forces 模块
forces = cace.modules.Forces(
    energy_key="CACE_energy",
    forces_key="CACE_forces",
    calc_stress=False,
)
# 带 Forces 的完整模型
model_f = NeuralNetworkPotential(
    input_modules=None,
    representation=rep,
    output_modules=[sr_energy, chi, system_charge_from_q, charge_eq, e_add, forces],
).to(device)
# 加载 “最佳 MAE e 模型”：N_18_BSA_min_mae_e20260312_154625_model.pth
# min_mae_path = DATA_DIR / "loss_data" / "N_18_BSA_min_mae_e20260312_154625_model.pth"
min_mae_path = DATA_DIR / "best_model.pth"
assert min_mae_path.exists(), f"{min_mae_path} not found"
print("Loading min-MAE-e model from:", min_mae_path)
obj = torch.load(str(min_mae_path), map_location=device, weights_only=False)
if isinstance(obj, dict) and "model_state_dict" in obj:
    state_dict = obj["model_state_dict"]
    print("Loaded checkpoint dict (model_state_dict).")
else:
    state_dict = obj.state_dict()
    print("Loaded full model object; using its state_dict.")
missing, unexpected = model_f.load_state_dict(state_dict, strict=False)
print("missing keys:", len(missing), "unexpected keys:", len(unexpected))
model_f.to(device)
model_f.eval()
# 数据集（与训练脚本完全一致）
collection_f = get_dataset_from_extxyz_with_charge(
    train_path=str(DATA_DIR / "NaCl.xyz"),
    cutoff=cutoff,
    valid_fraction=0.1,
    seed=1,
    atomic_energies={11: -4417.07609365649, 17: -12516.880649933015},
)
train_dataset_f = collection_f.train
valid_dataset_f = collection_f.valid
train_loader_f = torch_geometric.DataLoader(
    train_dataset_f,
    batch_size=5,
    shuffle=True,
    drop_last=True,
)
valid_loader_f = torch_geometric.DataLoader(
    valid_dataset_f,
    batch_size=5,
    shuffle=False,
    drop_last=False,
)
print("train samples:", len(train_dataset_f))
print("valid samples:", len(valid_dataset_f))

Loading min-MAE-e model from: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq/fit-4hdnnp-NaCl-initial/best_model.pth
Loaded full model object; using its state_dict.
missing keys: 0 unexpected keys: 12
train samples: 4500
valid samples: 500


In [3]:
# Metrics 统计（与 log_5197563 定义一致：能量 per_atom）
e_metric_train = Metrics(
    target_name="energy",
    predict_name="CACE_energy",
    name="e/atom",
    per_atom=True,
)
e_metric_valid = Metrics(
    target_name="energy",
    predict_name="CACE_energy",
    name="e/atom",
    per_atom=True,
)
f_metric_train = Metrics(
    target_name="forces",
    predict_name="CACE_forces",
    name="f",
)
f_metric_valid = Metrics(
    target_name="forces",
    predict_name="CACE_forces",
    name="f",
)
# 同时收集扁平的 E/F 误差（便于核实）
all_e_pred_train, all_e_ref_train = [], []
all_e_pred_valid, all_e_ref_valid = [], []
all_f_pred_train, all_f_ref_train = [], []
all_f_pred_valid, all_f_ref_valid = [], []
# 训练集
for batch in train_loader_f:
    batch = batch.to(device)
    batch_dict = batch.to_dict()
    if "positions" in batch_dict:
        batch_dict["positions"] = batch_dict["positions"].detach().requires_grad_(True)
    out = model_f(batch_dict, training=False)
    # Metrics
    e_metric_train.update_metrics(subset="train", pred=out, target=batch_dict)
    if "CACE_forces" in out and "forces" in batch_dict:
        f_metric_train.update_metrics(subset="train", pred=out, target=batch_dict)
    # 简单统计
    e_pred = out["CACE_energy"].view(-1)
    e_ref = batch_dict["energy"].view(-1)
    all_e_pred_train.append(e_pred.detach().cpu().numpy())
    all_e_ref_train.append(e_ref.detach().cpu().numpy())
    if "CACE_forces" in out and "forces" in batch_dict:
        f_pred = out["CACE_forces"].view(-1, 3)
        f_ref = batch_dict["forces"].view(-1, 3)
        all_f_pred_train.append(f_pred.detach().cpu().numpy())
        all_f_ref_train.append(f_ref.detach().cpu().numpy())
# 验证集
for batch in valid_loader_f:
    batch = batch.to(device)
    batch_dict = batch.to_dict()
    if "positions" in batch_dict:
        batch_dict["positions"] = batch_dict["positions"].detach().requires_grad_(True)
    out = model_f(batch_dict, training=False)
    e_metric_valid.update_metrics(subset="val", pred=out, target=batch_dict)
    if "CACE_forces" in out and "forces" in batch_dict:
        f_metric_valid.update_metrics(subset="val", pred=out, target=batch_dict)
    e_pred = out["CACE_energy"].view(-1)
    e_ref = batch_dict["energy"].view(-1)
    all_e_pred_valid.append(e_pred.detach().cpu().numpy())
    all_e_ref_valid.append(e_ref.detach().cpu().numpy())
    if "CACE_forces" in out and "forces" in batch_dict:
        f_pred = out["CACE_forces"].view(-1, 3)
        f_ref = batch_dict["forces"].view(-1, 3)
        all_f_pred_valid.append(f_pred.detach().cpu().numpy())
        all_f_ref_valid.append(f_ref.detach().cpu().numpy())
print("=== Metrics (对齐 log_5197563 定义) ===")
train_e = e_metric_train.retrieve_metrics(subset="train", clear=False, print_log=False)
val_e   = e_metric_valid.retrieve_metrics(subset="val", clear=False, print_log=False)
print("Train e/atom MAE, RMSE (eV):", float(train_e["mae"]), float(train_e["rmse"]))
print("Valid e/atom MAE, RMSE (eV):", float(val_e["mae"]), float(val_e["rmse"]))
if len(f_metric_train.logs["train"]["pred"]):
    train_f = f_metric_train.retrieve_metrics(subset="train", clear=False, print_log=False)
    val_f   = f_metric_valid.retrieve_metrics(subset="val", clear=False, print_log=False)
    print("Train f MAE, RMSE (eV/Å):", float(train_f["mae"]), float(train_f["rmse"]))
    print("Valid f MAE, RMSE (eV/Å):", float(val_f["mae"]), float(val_f["rmse"]))
else:
    print("[Warning] 当前模型输出中没有 'CACE_forces'，无法计算力的 MAE/RMSE。")
# 再用简单方式核对一次（每结构 MAE/RMSE）
print("\n=== Simple per-structure stats (check) ===")
E_pred_train = np.concatenate(all_e_pred_train)
E_ref_train  = np.concatenate(all_e_ref_train)
E_pred_valid = np.concatenate(all_e_pred_valid)
E_ref_valid  = np.concatenate(all_e_ref_valid)
err_E_train = E_pred_train - E_ref_train
err_E_valid = E_pred_valid - E_ref_valid
print("Train Energy MAE, RMSE (eV):", np.abs(err_E_train).mean(), np.sqrt((err_E_train**2).mean()))
print("Valid Energy MAE, RMSE (eV):", np.abs(err_E_valid).mean(), np.sqrt((err_E_valid**2).mean()))
if len(all_f_pred_train) and len(all_f_pred_valid):
    F_pred_train = np.concatenate(all_f_pred_train, axis=0)
    F_ref_train  = np.concatenate(all_f_ref_train, axis=0)
    F_pred_valid = np.concatenate(all_f_pred_valid, axis=0)
    F_ref_valid  = np.concatenate(all_f_ref_valid, axis=0)
    err_F_train = F_pred_train - F_ref_train
    err_F_valid = F_pred_valid - F_ref_valid
    print("Train Forces MAE, RMSE (eV/Å):", np.abs(err_F_train).mean(), np.sqrt((err_F_train**2).mean()))
    print("Valid Forces MAE, RMSE (eV/Å):", np.abs(err_F_valid).mean(), np.sqrt((err_F_valid**2).mean()))
else:
    print("No forces found in outputs / targets.")

=== Metrics (对齐 log_5197563 定义) ===
Train e/atom MAE, RMSE (eV): 0.8874984979629517 0.8878069519996643
Valid e/atom MAE, RMSE (eV): 0.8860527276992798 0.8863617181777954
Train f MAE, RMSE (eV/Å): 0.13087327778339386 0.20701420307159424
Valid f MAE, RMSE (eV/Å): 0.12956182658672333 0.20485830307006836

=== Simple per-structure stats (check) ===
Train Energy MAE, RMSE (eV): 14.62925 14.629542
Valid Energy MAE, RMSE (eV): 14.6349535 14.635243
Train Forces MAE, RMSE (eV/Å): 0.13087328 0.20701419
Valid Forces MAE, RMSE (eV/Å): 0.12956183 0.2048583


In [ ]:
# 在一个小批次上打印数据中的能量/力与模型预测值，方便逐项对比

# 取训练集和验证集各一个 batch 做示例
from itertools import islice

# 确保 model_f、train_loader_f、valid_loader_f 已经在上面的 cell 中构建好并运行

model_f.eval()

def show_batch_energy_forces(loader, name):
    batch = next(islice(iter(loader), 0, 1))  # 取第 0 个 batch
    batch = batch.to(device)
    batch_dict = batch.to_dict()
    # Forces 需要 positions 有梯度
    if "positions" in batch_dict:
        batch_dict["positions"] = batch_dict["positions"].detach().requires_grad_(True)

    with torch.enable_grad():
        out = model_f(batch_dict, training=False)

    E_pred = out["CACE_energy"].detach().cpu().numpy().reshape(-1)
    E_ref  = batch_dict["energy"].detach().cpu().numpy().reshape(-1)

    F_pred = out.get("CACE_forces", None)
    F_ref  = batch_dict.get("forces", None)

    print(f"\n=== {name} batch energy ===")
    for i, (ep, er) in enumerate(zip(E_pred, E_ref)):
        print(f"struct {i}: E_ref = {er:.6f} eV, E_pred = {ep:.6f} eV, diff = {ep-er:.6f} eV")

    if F_pred is not None and F_ref is not None:
        F_pred_np = F_pred.detach().cpu().numpy().reshape(-1, 3)
        F_ref_np  = F_ref.detach().cpu().numpy().reshape(-1, 3)
        print(f"\n=== {name} batch forces (前 10 个原子) ===")
        n_show = min(10, F_pred_np.shape[0])
        for i in range(n_show):
            fx_p, fy_p, fz_p = F_pred_np[i]
            fx_r, fy_r, fz_r = F_ref_np[i]
            print(
                f"atom {i}: F_ref = ({fx_r:.4f}, {fy_r:.4f}, {fz_r:.4f}) eV/Å, "
                f"F_pred = ({fx_p:.4f}, {fy_p:.4f}, {fz_p:.4f}) eV/Å"
            )
    else:
        print(f"\n[{name}] 当前 batch 中缺少 forces 或模型未输出 CACE_forces，无法打印力对比。")

# 打印训练集与验证集各一个 batch 的对比
show_batch_energy_forces(train_loader_f, "Train")
show_batch_energy_forces(valid_loader_f, "Valid")